# 01 — Data Collection
Pulls World Bank demographic/economic indicators and OWID energy data for 8 case-study countries (1960–2024).
**Output:** `population_demographics.csv`, `energy_data.csv`

In [4]:
!pip install -q requests pandas

import requests
import pandas as pd
import time

COUNTRIES = {
    "JPN": "Japan", "DEU": "Germany", "CHN": "China", "NGA": "Nigeria",
    "IND": "India", "IDN": "Indonesia", "USA": "United States", "BRA": "Brazil",
}
COUNTRY_CODES = ";".join(COUNTRIES.keys())

INDICATORS = {
    "SP.POP.TOTL": "population_total",
    "SP.POP.GROW": "population_growth_pct",
    "SP.POP.DPND": "age_dependency_ratio",
    "SP.POP.65UP.TO.ZS": "pop_65_plus_pct",
    "SP.POP.1564.TO.ZS": "pop_working_age_pct",
    "SP.URB.TOTL.IN.ZS": "urban_pop_pct",
    "NY.GDP.PCAP.CD": "gdp_per_capita_usd",
    "NY.GDP.MKTP.KD.ZG": "gdp_growth_pct",
}
YEAR_RANGE = "1960:2024"

In [12]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def get_session():
    session = requests.Session()
    retry = Retry(total=3, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    return session

def fetch_indicator(indicator_code, indicator_name):
    url = (
        f"https://api.worldbank.org/v2/country/{COUNTRY_CODES}"
        f"/indicator/{indicator_code}?format=json&per_page=2000&date={YEAR_RANGE}"
    )
    session = get_session()
    resp = session.get(url, timeout=120)  # increased from 30 to 120
    resp.raise_for_status()
    payload = resp.json()
    if len(payload) < 2 or payload[1] is None:
        print(f"  WARNING: no data returned for {indicator_code}")
        return pd.DataFrame()
    rows = []
    for r in payload[1]:
        rows.append({
            "country_code": r["countryiso3code"],
            "country": r["country"]["value"],
            "year": int(r["date"]),
            indicator_name: r["value"],
        })
    return pd.DataFrame(rows)

def build_population_dataset():
    print("Fetching World Bank indicators...")
    dfs = []
    for code, name in INDICATORS.items():
        print(f"  - {name} ({code})")
        df = fetch_indicator(code, name)
        if not df.empty:
            dfs.append(df.set_index(["country_code", "country", "year"]))
        time.sleep(0.3)
    merged = pd.concat(dfs, axis=1).reset_index()
    return merged

def fetch_energy_data():
    print("Fetching Our World in Data energy dataset...")
    url = "https://nyc3.digitaloceanspaces.com/owid-public/data/energy/owid-energy-data.csv"
    df = pd.read_csv(url)
    df = df[df["iso_code"].isin(COUNTRIES.keys())]
    df = df[(df["year"] >= 1965) & (df["year"] <= 2024)]
    return df

In [13]:
pop_df = build_population_dataset()
pop_df.to_csv("population_demographics.csv", index=False)
print(f"Saved population_demographics.csv ({pop_df.shape[0]} rows)")

energy_df = fetch_energy_data()
energy_df.to_csv("energy_data.csv", index=False)
print(f"Saved energy_data.csv ({energy_df.shape[0]} rows)")

print("\n--- population_demographics.csv preview ---")
display(pop_df.head())
print("\n--- energy_data.csv preview ---")
display(energy_df.head())

Fetching World Bank indicators...
  - population_total (SP.POP.TOTL)
  - population_growth_pct (SP.POP.GROW)
  - age_dependency_ratio (SP.POP.DPND)
  - pop_65_plus_pct (SP.POP.65UP.TO.ZS)
  - pop_working_age_pct (SP.POP.1564.TO.ZS)
  - urban_pop_pct (SP.URB.TOTL.IN.ZS)
  - gdp_per_capita_usd (NY.GDP.PCAP.CD)
  - gdp_growth_pct (NY.GDP.MKTP.KD.ZG)


Saved population_demographics.csv (520 rows)
Fetching Our World in Data energy dataset...
Saved energy_data.csv (480 rows)

--- population_demographics.csv preview ---


,country_code,country,year,population_total,population_growth_pct,age_dependency_ratio,pop_65_plus_pct,pop_working_age_pct,urban_pop_pct,gdp_per_capita_usd,gdp_growth_pct
0,BRA,Brazil,2024,211998573,0.405467,44.329976,11.049184,69.285676,87.895855,10310.548697,3.419315
1,BRA,Brazil,2023,211140729,0.395929,44.029095,10.633631,69.430416,87.615971,10377.589279,3.241655
2,BRA,Brazil,2022,210306415,0.360181,43.790263,10.251785,69.545738,87.349556,9281.333344,3.016694
3,BRA,Brazil,2021,209550294,0.425361,43.640470,9.911100,69.618263,87.136078,7972.536650,4.762604
4,BRA,Brazil,2020,208660842,0.579351,43.586067,9.598665,69.644640,86.911219,7074.193783,-3.276759



--- energy_data.csv preview ---


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
3033,Brazil,1965,BRA,83817535.0,3.005470e+11,NaN,NaN,0.0,0.0,NaN,...,NaN,0.0,NaN,NaN,0.0,0.0,0.0,0.0,NaN,0.0
3034,Brazil,1966,BRA,86139306.0,3.205837e+11,NaN,0.0,0.0,0.0,NaN,...,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0
3035,Brazil,1967,BRA,88446075.0,3.341491e+11,NaN,0.0,0.0,0.0,NaN,...,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0
3036,Brazil,1968,BRA,90741191.0,3.668048e+11,NaN,0.0,0.0,0.0,NaN,...,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0
3037,Brazil,1969,BRA,93045719.0,4.016017e+11,NaN,0.0,0.0,0.0,NaN,...,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0


## Upload UN WPP files
Download these manually from https://ourworldindata.org/grapher/population-long-run-with-projections and https://ourworldindata.org/grapher/population-by-age-group-with-projections, then upload below.

In [15]:
from google.colab import files

# Upload population-long-run-with-projections file
uploaded = files.upload()
proj_filename = list(uploaded.keys())[0]
proj_raw = pd.read_csv(proj_filename)
proj_raw.to_csv("population_projections.csv", index=False)
print(f"Saved population_projections.csv — columns: {proj_raw.columns.tolist()}")

# Upload population-by-age-group-with-projections file
uploaded2 = files.upload()
age_filename = list(uploaded2.keys())[0]
pop_age_proj = pd.read_csv(age_filename)
pop_age_proj.to_csv("population_age_projections.csv", index=False)
print(f"Saved population_age_projections.csv — columns: {pop_age_proj.columns.tolist()}")

Saving population-long-run-with-projections.csv to population-long-run-with-projections.csv
Saved population_projections.csv — columns: ['Entity', 'Code', 'Year', 'Population (projections) (Projected)', 'Population']


Saving population_age_projections.csv to population_age_projections.csv
Saved population_age_projections.csv — columns: ['Entity', 'Code', 'Year', 'Total', 'Total (Projected)', 'Ages 65+', 'Ages 65+ (Projected)', 'Ages 25-64', 'Ages 25-64 (Projected)', 'Under-25s (Projected)', 'Under-25s', 'Under-15s', 'Under-15s (Projected)', 'Under-5s', 'Under-5s (Projected)']
